### Import *p*-less and *p*-less<sub>norm</sub> Samplers

In [1]:
from p_less_samplers import p_less_decode, p_less_norm_decode

help(p_less_decode)
help(p_less_norm_decode)

Help on function p_less_decode in module p_less_samplers:

p_less_decode(probs: torch.Tensor) -> torch.Tensor
    Perform p-less sampling on a token probability distribution. Takes in a probability distribution over the
    vocabulary and returns the sampled token index.
    
    Args:
        probs (torch.Tensor): Probability distribution over the vocabulary, shape: (batch_size, vocabulary_size)
    
    Returns:
        torch.Tensor: Sampled token index, shape: (batch_size, 1)
    
    Note:
        p-less sampling admits tokens whose probability mass is at least the p-less threshold. The p-less
        threshold is hyperparameter-free and derived from the full distribution, therefore no hyperparameter
        tuning is required. The resulting distribution is then re-normalized based on the admitted tokens, after
        which sampling is performed. p-less is bounded and valid, i.e. at least one token will satisfy this
        constraint, therefore there is no need to defend against 

### Example for *p*-less Decoding

In [2]:
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

os.environ["HF_TOKEN"] = ""  # Your access token to Hugging Face models
model_id = "meta-llama/Meta-Llama-3-8B-Instruct"
context = 'Solve 1 + 2 - 3 * 4 / 5. End with "The answer is [ANSWER]."'
max_generated_tokens = 512

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, use_cache=True, device_map="auto")
encodings = tokenizer.encode(context, return_tensors="pt").to(model.device)
past_key_values = None
generated_token_ids = []

with torch.no_grad():
    for _ in range(max_generated_tokens):
        output = model(input_ids=encodings, past_key_values=past_key_values, return_dict=True)

        logits = output.logits[0, -1]
        probs = torch.softmax(logits, dim=-1)
        # Use p-less to decode the next token
        next_token_id = p_less_decode(probs)
        generated_token_ids += next_token_id.tolist()

        if next_token_id == tokenizer.eos_token_id:
            break
        else:
            past_key_values = output.past_key_values
            encodings = next_token_id.unsqueeze(0)
            

print(f"Context: {context}")
print(f"Generated text: {tokenizer.decode(generated_token_ids)}")

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Context: Solve 1 + 2 - 3 * 4 / 5. End with "The answer is [ANSWER]."
Generated text:  

Note: The order of operations is to perform multiplication and division from left to right, then perform addition and subtraction from left to right. 

1 + 2 - 3 * 4 / 5 =?

First, perform multiplication and division from left to right:
3 * 4 = 12
1 + 2 - 12 / 5 =?

Next, perform division:
12 / 5 = 2.4
1 + 2 - 2.4 =?

Finally, perform addition and subtraction from left to right:
1 + 2 = 3
3 - 2.4 = 0.6

The answer is 0.6. <|eot_id|>


### Example for *p*-less<sub>norm</sub> Decoding

In [3]:
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

os.environ["HF_TOKEN"] = ""  # Your access token to Hugging Face models
model_id = "meta-llama/Meta-Llama-3-8B-Instruct"
context = "In a short summary, Large Language Model decoding is"
max_generated_tokens = 512

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, use_cache=True, device_map="auto")
encodings = tokenizer.encode(context, return_tensors="pt").to(model.device)
past_key_values = None
generated_token_ids = []

with torch.no_grad():
    for _ in range(max_generated_tokens):
        output = model(input_ids=encodings, past_key_values=past_key_values, return_dict=True)

        logits = output.logits[0, -1]
        probs = torch.softmax(logits, dim=-1)
        # Use p-less-norm to decode the next token
        next_token_id = p_less_norm_decode(probs)
        generated_token_ids += next_token_id.tolist()

        if next_token_id == tokenizer.eos_token_id:
            break
        else:
            past_key_values = output.past_key_values
            encodings = next_token_id.unsqueeze(0)
            

print(f"Context: {context}")
print(f"Generated text: {tokenizer.decode(generated_token_ids)}")

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Context: In a short summary, Large Language Model decoding is
Generated text:  the process of generating text based on the input given to the model. The model uses its internal knowledge and patterns learned from the training data to generate the output text. The quality of the output text depends on the quality of the input, the complexity of the task, and the capabilities of the model.

Here are some key points to consider when working with Large Language Models for decoding:

1. **Input quality**: The quality of the input text can significantly impact the quality of the output text. Make sure the input is clear, concise, and relevant to the task at hand.
2. **Model capabilities**: Different models have different capabilities and limitations. Choose a model that is suitable for the task at hand and has the necessary capabilities to generate high-quality output.
3. **Task complexity**: The complexity of the task can also impact the quality of the output text. Simple tasks like generat